In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated,Literal
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,BaseMessage
from langgraph.graph import add_messages
from langgraph.checkpoint.memory import MemorySaver



In [2]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [3]:
graph = StateGraph(ChatState)

In [4]:
model=ChatOllama(model="llama3.1:8b")


In [5]:
def chat_node(state:ChatState):
    messages = state['messages']
    response = model.invoke(messages)
    return {'messages': [response]}

In [6]:
checkpointer = MemorySaver()
graph.add_node('chat-node', chat_node)

graph.add_edge(START, 'chat-node')
graph.add_edge('chat-node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [8]:
thread_id =1
while True:
    user_message=input("Type here: ")
    print('User:', user_message)

    if user_message.strip().lower() in ['exit', 'quit']:
        break
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config={'configurable': {'thread_id': thread_id}})
    print('Chatbot:', response['messages'][-1].content)


User: hi
Chatbot: It's nice to meet you. Is there something I can help you with or would you like to chat?
User: my name is abhi
Chatbot: Nice to meet you, Abhi! How's your day going so far?
User: what is my name 
Chatbot: Your name is ABHI.
User: do 3 +78
Chatbot: The answer is: 81
User: ok then multiply is by 2
Chatbot: Multiplying 81 by 2 gives:

162
User: quit
